In [19]:
from pm4py.objects.log.importer.xes import importer as xes_importer
import numpy as np
import json
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

log = xes_importer.apply("data/financial_log.xes")

traces = []

for trace in log:
    acts = [event["concept:name"] for event in trace]
    if len(acts) > 1:
        traces.append(acts)

print("Total traces:", len(traces))

parsing log, completed traces ::   0%|          | 0/13087 [00:00<?, ?it/s]

Total traces: 13087


In [20]:
lengths = [len(t) for t in traces]
threshold = np.percentile(lengths, 95)

filtered_traces = [t for t in traces if len(t) <= threshold]

print("After filtering:", len(filtered_traces))


After filtering: 12454


In [21]:
def collapse_loops(trace):
    new_trace = [trace[0]]
    for act in trace[1:]:
        if act != new_trace[-1]:
            new_trace.append(act)
    return new_trace

processed_traces = [collapse_loops(t) for t in filtered_traces]

print("After loop collapse:", len(processed_traces))

After loop collapse: 12454


In [22]:
case_ids = list(range(len(processed_traces)))

train_ids, test_ids = train_test_split(
    case_ids,
    test_size=0.2,
    random_state=SEED
)

train_traces = [processed_traces[i] for i in train_ids]
test_traces  = [processed_traces[i] for i in test_ids]

print("Train:", len(train_traces))
print("Test:", len(test_traces))

Train: 9963
Test: 2491


In [23]:
activities = sorted(set(a for t in train_traces for a in t))

activity_to_id = {a: i for i, a in enumerate(activities)}
id_to_activity = {i: a for a, i in activity_to_id.items()}

print("Total activities:", len(activity_to_id))

Total activities: 24


In [24]:
def encode_trace(trace):
    return [activity_to_id[a] for a in trace]

train_encoded = [encode_trace(t) for t in train_traces]
test_encoded  = [encode_trace(t) for t in test_traces]

In [25]:
# activity frequency
all_acts = [a for t in train_traces for a in t]
act_counts = Counter(all_acts)

# bigram distribution
bigram_counts = Counter()
for t in train_traces:
    for i in range(len(t)-1):
        bigram_counts[(t[i], t[i+1])] += 1

total_bigrams = sum(bigram_counts.values())
bigram_probs = {k: v/total_bigrams for k,v in bigram_counts.items()}

In [26]:
rare_acts = [a for a, c in act_counts.items() if c < np.percentile(list(act_counts.values()), 30)]

def inject_semantic(trace):
    t = trace.copy()
    idx = np.random.randint(len(t))
    t[idx] = np.random.choice(rare_acts)
    return t

In [27]:
def inject_control(trace):
    t = trace.copy()
    if len(t) < 3:
        return t
    
    i = np.random.randint(0, len(t)-1)
    t[i], t[i+1] = t[i+1], t[i]
    return t

In [28]:
def inject_temporal(trace):
    t = trace.copy()
    if len(t) < 4:
        return t
    
    i = np.random.randint(0, len(t)-2)
    t[i:i+3] = reversed(t[i:i+3])
    return t

In [29]:
test_mca = []
labels = []
types = []

for t in test_traces:
    
    # keep normal
    test_mca.append(t)
    labels.append(0)
    types.append("N")
    
    # semantic
    test_mca.append(inject_semantic(t))
    labels.append(1)
    types.append("S")
    
    # control-flow
    test_mca.append(inject_control(t))
    labels.append(1)
    types.append("C")
    
    # temporal
    test_mca.append(inject_temporal(t))
    labels.append(1)
    types.append("T")

In [30]:
test_mca_encoded = [
    [activity_to_id[a] for a in trace if a in activity_to_id]
    for trace in test_mca
]

In [31]:
import os
os.makedirs("bpi_mca_dataset", exist_ok=True)

# JSON (string)
json.dump(train_traces, open("bpi_mca_dataset/X_train.json", "w"))
json.dump(test_mca, open("bpi_mca_dataset/X_test.json", "w"))

# Encoded
json.dump(train_encoded, open("bpi_mca_dataset/X_train_encoded.json", "w"))
json.dump(test_mca_encoded, open("bpi_mca_dataset/X_test_encoded.json", "w"))

# Labels
pd.DataFrame({
    "label": labels,
    "type": types
}).to_csv("bpi_mca_dataset/y_test.csv", index=False)

# Mapping
json.dump(activity_to_id, open("bpi_mca_dataset/activity_to_id.json", "w"))
json.dump(id_to_activity, open("bpi_mca_dataset/id_to_activity.json", "w"))

print("✅ BPI MCA dataset created")

✅ BPI MCA dataset created


In [33]:
import json
import pandas as pd
import numpy as np

DATA_DIR = "bpi_mca_dataset"

X_train = json.load(open(f"{DATA_DIR}/X_train.json"))
X_test  = json.load(open(f"{DATA_DIR}/X_test.json"))

df = pd.read_csv(f"{DATA_DIR}/y_test.csv")

labels = df["label"].values
types  = df["type"].values

print("Train:", len(X_train))
print("Test:", len(X_test))

from collections import Counter

print("\n📊 Label Distribution:")
print(Counter(labels))

print("\n📊 Type Distribution:")
print(Counter(types))

Train: 9963
Test: 9964

📊 Label Distribution:
Counter({1: 7473, 0: 2491})

📊 Type Distribution:
Counter({'N': 2491, 'S': 2491, 'C': 2491, 'T': 2491})


In [34]:
print("Sample anomaly types:")
for i in range(10):
    print(types[i], labels[i])

Sample anomaly types:
N 0
S 1
C 1
T 1
N 0
S 1
C 1
T 1
N 0
S 1
